In [4]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy import stats
from statsmodels.stats.multitest import multipletests

results_dir = Path("results/posterior_csvs")

def load_posterior(input_vo, results_dir=results_dir):
    """Load posterior samples for a given input_vo."""
    path = results_dir / f"posterior_samples_{input_vo}.csv"
    if not path.exists():
        raise FileNotFoundError(f"No posterior found for {input_vo} at {path}")
    df = pd.read_csv(path)
    # Parameter names are all columns except input_vo label column
    param_names = [c for c in df.columns if c != 'input_vo']
    return df, param_names


def compare_posteriors_clt(df_a, df_b, label_a, label_b, param_names, alpha=0.05):
    """
    CLT-based two-sample Welch t-test between two posterior distributions.
    Automatically uses whatever parameters are available in both groups.
    """
    results = []
    for param in param_names:
        a = df_a[param].dropna().values
        b = df_b[param].dropna().values

        mean_a, mean_b = np.mean(a), np.mean(b)
        median_a, median_b = np.median(a), np.median(b)
        se_a = np.std(a, ddof=1) / np.sqrt(len(a))
        se_b = np.std(b, ddof=1) / np.sqrt(len(b))

        t_stat, p_val = stats.ttest_ind(a, b, equal_var=False)

        pooled_std = np.sqrt((np.std(a, ddof=1)**2 + np.std(b, ddof=1)**2) / 2)
        cohens_d = (mean_a - mean_b) / pooled_std if pooled_std > 0 else np.nan

        diff = mean_a - mean_b
        se_diff = np.sqrt(se_a**2 + se_b**2)

        results.append({
            'Parameter'          : param,
            f'{label_a}_mean'    : mean_a,
            f'{label_a}_median'  : median_a,
            f'{label_a}_SE'      : se_a,
            f'{label_b}_mean'    : mean_b,
            f'{label_b}_median'  : median_b,
            f'{label_b}_SE'      : se_b,
            'Difference'         : diff,
            'CI_low'             : diff - 1.96 * se_diff,
            'CI_high'            : diff + 1.96 * se_diff,
            't_stat'             : t_stat,
            'p_value'            : p_val,
            'Cohens_d'           : cohens_d,
        })

    df_results = pd.DataFrame(results)

    # FDR correction across all tests
    _, p_adj, _, _ = multipletests(df_results['p_value'], method='fdr_bh')
    df_results['p_adjusted'] = p_adj
    df_results['Significant'] = p_adj < alpha

    return df_results


def compare_all_ages(ages, results_dir=results_dir, alpha=0.05):
    """
    Run Hx vs Nx comparison for each age and combine into one table.
    Automatically finds shared parameters between groups.
    """
    all_results = []

    for age in ages:
        try:
            df_hx, params_hx = load_posterior(f"{age}_Hx", results_dir)
            df_nx, params_nx = load_posterior(f"{age}_Nx", results_dir)
        except FileNotFoundError as e:
            print(f"Skipping {age}: {e}")
            continue

        # Use only parameters present in both groups
        shared_params = [p for p in params_hx if p in params_nx]
        print(f"{age} — comparing parameters: {shared_params}")

        res = compare_posteriors_clt(
            df_hx, df_nx,
            label_a=f"Hx_{age}",
            label_b=f"Nx_{age}",
            param_names=shared_params,
            alpha=alpha
        )
        res.insert(0, 'Age', age)
        all_results.append(res)

    df_final = pd.concat(all_results, ignore_index=True)

    # Re-correct p-values across ALL tests (all ages × all params)
    _, p_adj_global, _, _ = multipletests(df_final['p_value'], method='fdr_bh')
    df_final['p_adjusted_global'] = p_adj_global
    df_final['Significant_global'] = p_adj_global < alpha

    return df_final

def classify_effect_size(d):
    """Classify Cohen's d by magnitude."""
    if pd.isna(d):
        return 'NA'
    d = abs(d)
    if d < 0.2:
        return 'negligible'
    elif d < 0.5:
        return 'small'
    elif d < 0.8:
        return 'medium'
    else:
        return 'large'

# Apply classification
df_final['Effect_size'] = df_final['Cohens_d'].apply(classify_effect_size)

# ── Run the comparison ────────────────────────────────────────────────────────
ages = ['P21', 'P90', 'P365']
df_final = compare_all_ages(ages)

# Save
df_final.to_csv(results_dir / "statistical_comparison.csv", index=False)

# Print clean summary
summary_cols = ['Age', 'Parameter', 
                'Hx_P21_mean', 'Nx_P21_mean',   # these will vary by age
                'Difference', 'CI_low', 'CI_high',
                'p_value', 'p_adjusted_global', 'Cohens_d', 'Significant_global']
available_cols = [c for c in summary_cols if c in df_final.columns]
print(df_final.to_string(index=False))

P21 — comparing parameters: ['AmRefRfw', 'Cvs', 'Rap', 'sfact', 'c3']
P90 — comparing parameters: ['AmRefRfw', 'Cvs', 'Rap', 'sfact', 'c3']
P365 — comparing parameters: ['AmRefRfw', 'Cvs', 'Rap', 'sfact', 'c3']
 Age Parameter  Hx_P21_mean  Hx_P21_median  Hx_P21_SE  Nx_P21_mean  Nx_P21_median  Nx_P21_SE  Difference     CI_low   CI_high     t_stat      p_value  Cohens_d   p_adjusted  Significant  Hx_P90_mean  Hx_P90_median  Hx_P90_SE  Nx_P90_mean  Nx_P90_median  Nx_P90_SE  Hx_P365_mean  Hx_P365_median  Hx_P365_SE  Nx_P365_mean  Nx_P365_median  Nx_P365_SE  p_adjusted_global  Significant_global
 P21  AmRefRfw   115.572282     115.185583   0.449295   125.661340     125.395050   0.450839  -10.089058 -11.336583 -8.841534 -15.851039 5.366734e-46 -1.427765 1.341684e-45         True          NaN            NaN        NaN          NaN            NaN        NaN           NaN             NaN         NaN           NaN             NaN         NaN       4.025051e-45                True
 P21       Cvs 

In [11]:
def classify_effect_size(d):
    """Classify Cohen's d by magnitude."""
    if pd.isna(d):
        return 'NA'
    d = abs(d)
    if d < 0.2:
        return 'negligible'
    elif d < 0.5:
        return 'small'
    elif d < 0.8:
        return 'medium'
    else:
        return 'large'

# Apply classification
df_final['Effect_size'] = df_final['Cohens_d'].apply(classify_effect_size)

# Save
df_final[['Age', 'Parameter', 'Cohens_d', 'Effect_size']].to_excel(
    results_dir / "cohens_d_results.xlsx", index=False
)
# Print summary
print(df_final[['Age', 'Parameter', 'Cohens_d', 'Effect_size']].to_string(index=False))

 Age Parameter  Cohens_d Effect_size
 P21  AmRefRfw -1.427765       large
 P21       Cvs -1.295512       large
 P21       Rap  1.442799       large
 P21     sfact  0.243983       small
 P21        c3  0.175828  negligible
 P90  AmRefRfw  1.382304       large
 P90       Cvs -1.110238       large
 P90       Rap  0.279863       small
 P90     sfact  0.155086  negligible
 P90        c3  0.167719  negligible
P365  AmRefRfw  0.762282      medium
P365       Cvs -0.786492      medium
P365       Rap  0.182935  negligible
P365     sfact  0.041013  negligible
P365        c3  0.116270  negligible


In [ ]:
import numpy as np
import pandas as pd

# Body weight means and SDs
bw = {
    'Hx_P21':  {'mean': 58.34,  'sd': 2.800799985},
    'Nx_P21':  {'mean': 57.46,  'sd': 1.24},
    'Hx_P90':  {'mean': 295.0,  'sd': 74.71117721},
    'Nx_P90':  {'mean': 273.5,  'sd': 63.37389052},
    'Hx_P365': {'mean': 431.5,  'sd': 108.1195612},
    'Nx_P365': {'mean': 438.0,  'sd': 107.235066},
}

# Posterior means and SDs from df_final (Hx - Nx per age)
params = {
    'P21':  {'AmRefRfw': {'Hx': (115.572282, 0.449295*np.sqrt(1000)), 'Nx': (125.661340, 0.450839*np.sqrt(1000))},
             'Cvs':      {'Hx': (0.150389,   0.003755*np.sqrt(1000)), 'Nx': (0.221871,   0.003262*np.sqrt(1000))},
             'Rap':      {'Hx': (120.914678,  2.141995*np.sqrt(1000)), 'Nx': (76.451985,  1.769741*np.sqrt(1000))}},
    'P90':  {'AmRefRfw': {'Hx': (350.407161, 1.382763*np.sqrt(1000)), 'Nx': (324.470329, 1.021210*np.sqrt(1000))},
             'Cvs':      {'Hx': (0.315079,   0.008294*np.sqrt(1000)), 'Nx': (0.489417,   0.011985*np.sqrt(1000))},
             'Rap':      {'Hx': (16.650856,  0.441092*np.sqrt(1000)), 'Nx': (14.716014,  0.460501*np.sqrt(1000))}},
    'P365': {'AmRefRfw': {'Hx': (475.310392, 1.794717*np.sqrt(1000)), 'Nx': (456.581865, 1.442791*np.sqrt(1000))},
             'Cvs':      {'Hx': (0.868335,   0.023387*np.sqrt(1000)), 'Nx': (1.101170,   0.015119*np.sqrt(1000))},
             'Rap':      {'Hx': (8.671789,   0.234981*np.sqrt(1000)), 'Nx': (8.013562,   0.239840*np.sqrt(1000))}},
}

def cohens_d_normalized(param_mean_hx, param_mean_nx, bw_hx, bw_nx):
    norm_hx = param_mean_hx / bw_hx['mean']
    norm_nx = param_mean_nx / bw_nx['mean']
    # Propagate uncertainty: SD of (param/BW) ≈ (param/BW) * sqrt((sd_param/mean_param)^2 + (sd_bw/mean_bw)^2)
    sd_norm_hx = norm_hx * np.sqrt((param_mean_hx[1]/param_mean_hx[0])**2 + (bw_hx['sd']/bw_hx['mean'])**2)
    sd_norm_nx = norm_nx * np.sqrt((param_mean_nx[1]/param_mean_nx[0])**2 + (bw_nx['sd']/bw_nx['mean'])**2)
    pooled_sd = np.sqrt((sd_norm_hx**2 + sd_norm_nx**2) / 2)
    d = (norm_hx - norm_nx) / pooled_sd
    return d